#### 1 : Import Libraries

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

#### 2 : Create Spark Session

In [2]:
import pyspark
print(pyspark.__version__)

4.2.0


In [3]:
import os
print(os.environ['JAVA_HOME'])

C:\Program Files\Java\jdk-26.0.1


In [4]:
import os
os.environ["JAVA_HOME"] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.19.10-hotspot"

opens = (
    "--add-opens=java.base/java.lang=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.invoke=ALL-UNNAMED "
    "--add-opens=java.base/java.lang.reflect=ALL-UNNAMED "
    "--add-opens=java.base/java.io=ALL-UNNAMED "
    "--add-opens=java.base/java.net=ALL-UNNAMED "
    "--add-opens=java.base/java.nio=ALL-UNNAMED "
    "--add-opens=java.base/java.util=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent=ALL-UNNAMED "
    "--add-opens=java.base/java.util.concurrent.atomic=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED "
    "--add-opens=java.base/sun.nio.cs=ALL-UNNAMED "
    "--add-opens=java.base/sun.security.action=ALL-UNNAMED "
    "--add-opens=java.base/sun.util.calendar=ALL-UNNAMED "
    "--add-opens=java.base/java.security=ALL-UNNAMED "
    "-Djdk.reflect.useDirectMethodHandle=false"
)

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--driver-java-options='{opens}' pyspark-shell"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Week5 Spark Basics")
    .config("spark.driver.extraJavaOptions", opens)
    .config("spark.executor.extraJavaOptions", opens)
    .getOrCreate()
)

print("Spark Session Created Successfully")

c:\Users\aditi\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


Spark Session Created Successfully


#### 3 : Load Dataset

In [5]:
df = spark.read.csv(
    "../data/dataset.csv",
    header=True,
    inferSchema=True
)

df.show(10)
df.printSchema()

+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|    customer_name|               email|          username|             city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1834|      2026-03-18|2025-07-25 05:27:09|    Jennifer Long| ftaylor@example.com|      jacobspamela|      East Rodney|         Texas|  India| South|     Electronics|     Mention|3561.0|       7|      14599|       7| 41|        Gold

In [6]:
print(df.columns)
print("Rows :", df.count())
print("Columns :", len(df.columns))

['user_id', 'transaction_date', 'raw_timestamp', 'customer_name', 'email', 'username', 'city', 'state', 'country', 'region', 'product_category', 'product_name', 'price', 'quantity', 'sale_amount', 'discount', 'age', 'subscription', 'status', 'store_id']
Rows : 25500
Columns : 20


#### 4 : Understanding Dataset

In [7]:
df.describe().show()

+-------+------------------+-------------+--------------------+----------+----------+-------+-------+------+----------------+------------+------------------+------------------+------------------+------------------+------------------+------------+--------+------------------+
|summary|           user_id|customer_name|               email|  username|      city|  state|country|region|product_category|product_name|             price|          quantity|       sale_amount|          discount|               age|subscription|  status|          store_id|
+-------+------------------+-------------+--------------------+----------+----------+-------+-------+------+----------------+------------+------------------+------------------+------------------+------------------+------------------+------------+--------+------------------+
|  count|             25500|        25500|               24251|     24145|     25500|  25500|  25500| 25500|           25500|       25500|             23502|             25500

#### 5 : Data Cleaning

In [8]:
print("Original Count :", df.count())

duplicate_removed = df.dropDuplicates()

print("After Removing Duplicates :", duplicate_removed.count())

Original Count : 25500
After Removing Duplicates : 25000


In [9]:
# Duplicates based on user_id and transaction_date
duplicate_removed2 = df.dropDuplicates(
    ["user_id","transaction_date"]
)

duplicate_removed2.show()

+-------+----------------+-------------------+------------------+--------------------+---------------+----------------+------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|     customer_name|               email|       username|            city|       state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+------------------+--------------------+---------------+----------------+------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1000|      2024-08-08|2026-06-20 15:28:55|  Charles Anderson|avilajulie@exampl...|    gabrielle73|      Scottshire|     Florida|  India|  East|     Electronics|        Term|  NULL|       2|      16133|      13| 43|     Premium|    NULL|      14|


In [10]:
# Null Values
from pyspark.sql.functions import col,isnan,when,count

df.select([
    count(
        when(col(c).isNull(),c)
    ).alias(c)
    for c in df.columns
]).show()

+-------+----------------+-------------+-------------+-----+--------+----+-----+-------+------+----------------+------------+-----+--------+-----------+--------+---+------------+------+--------+
|user_id|transaction_date|raw_timestamp|customer_name|email|username|city|state|country|region|product_category|product_name|price|quantity|sale_amount|discount|age|subscription|status|store_id|
+-------+----------------+-------------+-------------+-----+--------+----+-----+-------+------+----------------+------------+-----+--------+-----------+--------+---+------------+------+--------+
|      0|               0|            0|            0| 1249|    1355|   0|    0|      0|     0|               0|           0| 1998|       0|          0|       0|  0|           0|  8519|       0|
+-------+----------------+-------------+-------------+-----+--------+----+-----+-------+------+----------------+------------+-----+--------+-----------+--------+---+------------+------+--------+



In [11]:
# Drop Nulls
drop_null = df.na.drop()

drop_null.show()

+-------+----------------+-------------------+------------------+--------------------+---------------+--------------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|     customer_name|               email|       username|                city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+------------------+--------------------+---------------+--------------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1834|      2026-03-18|2025-07-25 05:27:09|     Jennifer Long| ftaylor@example.com|   jacobspamela|         East Rodney|         Texas|  India| South|     Electronics|     Mention|3561.0|       7|      14599|       7| 41|        

In [12]:
# Fill Null Values
fill_null = df.na.fill({
    "status":"Unknown",
    "price":0
})

fill_null.show()

+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|    customer_name|               email|          username|             city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1834|      2026-03-18|2025-07-25 05:27:09|    Jennifer Long| ftaylor@example.com|      jacobspamela|      East Rodney|         Texas|  India| South|     Electronics|     Mention|3561.0|       7|      14599|       7| 41|        Gold

In [13]:
# Remove Empty Username
clean_df = fill_null.filter(
    (col("email").isNotNull()) &
    (col("username") != "")
)

clean_df.show()

+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|    customer_name|               email|          username|             city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1834|      2026-03-18|2025-07-25 05:27:09|    Jennifer Long| ftaylor@example.com|      jacobspamela|      East Rodney|         Texas|  India| South|     Electronics|     Mention|3561.0|       7|      14599|       7| 41|        Gold

#### 6 : Filtering

In [14]:
# Premium Users
premium = clean_df.filter(
    col("subscription")=="Premium"
)

premium.show()

+-------+----------------+-------------------+------------------+--------------------+------------------+--------------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|     customer_name|               email|          username|                city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+------------------+--------------------+------------------+--------------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   2645|      2026-07-18|2025-02-11 11:49:54|       Luke Nelson|markrocha@example...|davenportchristine|   North Richardfort| New Hampshire|  India|  West|        Clothing|    Interest|   0.0|       8|      23306|      33|

In [15]:
# West Region
west = clean_df.filter(
    col("region")=="West"
)

west.show()

+-------+----------------+-------------------+------------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|     customer_name|               email|          username|             city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+------------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   2579|      2026-03-24|2026-05-16 05:37:16|      Sarah Weaver|patrick22@example...|     lindamartinez|        New Debra|      Illinois|  India|  West|       Furniture|        Soon|5855.0|       6|      22310|      28| 46|       B

In [16]:
# Age Between 18 and 30
young = clean_df.filter(
    (col("age")>=18) &
    (col("age")<=30)
)

young.show()

+-------+----------------+-------------------+----------------+--------------------+------------------+-----------------+-------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|   customer_name|               email|          username|             city|        state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+----------------+--------------------+------------------+-----------------+-------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1788|      2026-05-27|2024-08-13 09:43:06|       Sean Hood|clarkkaren@exampl...|       julieharris|        Hornmouth|New Hampshire|  India| South|     Electronics|        Firm| 969.0|       1|       3811|      16| 28|       Basic|  Activ

In [17]:
# Electronics Category
electronics = clean_df.filter(
    col("product_category")=="Electronics"
)

electronics.show()

+-------+----------------+-------------------+-----------------+--------------------+-------------+-------------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|      raw_timestamp|    customer_name|               email|     username|               city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+-----------------+--------------------+-------------+-------------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1834|      2026-03-18|2025-07-25 05:27:09|    Jennifer Long| ftaylor@example.com| jacobspamela|        East Rodney|         Texas|  India| South|     Electronics|     Mention|3561.0|       7|      14599|       7| 41|        Gold|  Active|  

#### 7 : Data Transformation

In [18]:
# Rename Column
rename_df = clean_df.withColumnRenamed(
    "raw_timestamp",
    "event_time"
)

rename_df.show()

+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|user_id|transaction_date|         event_time|    customer_name|               email|          username|             city|         state|country|region|product_category|product_name| price|quantity|sale_amount|discount|age|subscription|  status|store_id|
+-------+----------------+-------------------+-----------------+--------------------+------------------+-----------------+--------------+-------+------+----------------+------------+------+--------+-----------+--------+---+------------+--------+--------+
|   1834|      2026-03-18|2025-07-25 05:27:09|    Jennifer Long| ftaylor@example.com|      jacobspamela|      East Rodney|         Texas|  India| South|     Electronics|     Mention|3561.0|       7|      14599|       7| 41|        Gold

In [19]:
# Cast Timestamp
rename_df = rename_df.withColumn(
    "event_time",
    col("event_time").cast(TimestampType())
)

rename_df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: date (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- country: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- price: double (nullable = false)
 |-- quantity: integer (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- discount: integer (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- status: string (nullable = false)
 |-- store_id: integer (nullable = true)



#### 8 : Aggregation

In [20]:
# Total Rows
print(clean_df.count())

22958


In [21]:
# Average Price
clean_df.select(
    avg("price")
).show()

+-----------------+
|       avg(price)|
+-----------------+
|4658.438888404913|
+-----------------+



In [22]:
# Minimum Price
clean_df.select(
    min("price")
).show()

+----------+
|min(price)|
+----------+
|       0.0|
+----------+



In [23]:
# Maximum Price
clean_df.select(
    max("price")
).show()

+----------+
|max(price)|
+----------+
|   10000.0|
+----------+



In [24]:
# Multiple Aggregation
clean_df.agg(
    min("price").alias("Minimum Price"),
    max("price").alias("Maximum Price"),
    avg("price").alias("Average Price")
).show()

+-------------+-------------+-----------------+
|Minimum Price|Maximum Price|    Average Price|
+-------------+-------------+-----------------+
|          0.0|      10000.0|4658.438888404913|
+-------------+-------------+-----------------+



#### 9 : GroupBy

In [25]:
# City Count
clean_df.groupBy(
    "city"
).count().show()

+-----------------+-----+
|             city|count|
+-----------------+-----+
| Lake Danaborough|    1|
|      Lake Joshua|    6|
|      New Shannon|    4|
|        Millsview|    1|
|      Sydneyburgh|    1|
|       Jordanview|    1|
|New Jennifershire|    1|
|       Lowerybury|    1|
|     East Patrick|    3|
|       Denisefurt|    1|
|  East Wyattshire|    1|
|       Karenmouth|    3|
|       Petersbury|    1|
|      Tiffanyview|    3|
|      Port Monica|    2|
|        Paceburgh|    1|
|       East Tammy|    2|
|         Woodstad|    1|
|    Pattersonberg|    1|
|        Dianaland|    2|
+-----------------+-----+
only showing top 20 rows


In [26]:
# Store Revenue
clean_df.groupBy(
    "store_id"
).agg(
    sum("price").alias("Revenue")
).show()

+--------+---------+
|store_id|  Revenue|
+--------+---------+
|      31|2307437.0|
|      34|2456486.0|
|      28|2307168.0|
|      27|2078764.0|
|      26|2107077.0|
|      44|2098307.0|
|      12|2366067.0|
|      22|1997947.0|
|      47|1949813.0|
|       1|2050852.0|
|      13|2267740.0|
|      16|2181852.0|
|       6|2046018.0|
|       3|2005662.0|
|      20|2264110.0|
|      40|2053452.0|
|      48|2108889.0|
|       5|2046422.0|
|      19|1865097.0|
|      41|2095910.0|
+--------+---------+
only showing top 20 rows


In [27]:
# Average Sales
clean_df.groupBy(
    "product_category"
).agg(
    avg("sale_amount").alias("Average Sales")
).show()

+----------------+------------------+
|product_category|     Average Sales|
+----------------+------------------+
|          Sports|12737.462799223636|
|     Electronics|12726.030524220305|
|        Clothing|12761.224399911913|
|           Books|12914.941810344828|
|       Furniture|12717.949339683913|
+----------------+------------------+



In [28]:
# Cities with More Than 100 Records
clean_df.groupBy("city") \
    .count() \
    .filter("count > 100") \
    .show()

+----+-----+
|city|count|
+----+-----+
+----+-----+



#### 10 : Wide Transformation & Shuffle

##### Wide Transformation

A wide transformation is an operation in which data moves between different partitions of the cluster.

Examples:
- groupBy()
- join()
- distinct()

##### Shuffle

Shuffle is the redistribution of data across partitions during wide transformations.

Although shuffle enables grouping and aggregation, it increases network communication and may reduce performance.

#### 11 : Final Pipeline

In [29]:
pipeline = (
    df
    .dropDuplicates()
    .na.fill({
        "price":0,
        "status":"Unknown"
    })
    .filter(
        col("region")=="West"
    )
    .withColumnRenamed(
        "raw_timestamp",
        "event_time"
    )
    .withColumn(
        "event_time",
        col("event_time").cast(TimestampType())
    )
    .groupBy("store_id")
    .agg(
        sum("price").alias("Total Revenue")
    )
)

pipeline.show()

+--------+-------------+
|store_id|Total Revenue|
+--------+-------------+
|      31|     610941.0|
|      34|     666886.0|
|      28|     580017.0|
|      27|     541763.0|
|      26|     553094.0|
|      44|     599261.0|
|      12|     616727.0|
|      22|     656422.0|
|      47|     604190.0|
|       1|     587375.0|
|      13|     675084.0|
|      16|     565658.0|
|       6|     587120.0|
|       3|     558621.0|
|      20|     712110.0|
|      40|     605493.0|
|      48|     580645.0|
|       5|     590536.0|
|      19|     563462.0|
|      41|     512781.0|
+--------+-------------+
only showing top 20 rows
